In [75]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [76]:
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df = df.drop('customerID', axis=1)

In [77]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [78]:
df['TenureGroup'] = pd.cut(df['tenure'], bins=[0, 12, 24, 36, 48, 60, 72], 
                           labels=['0-1yr', '1-2yr', '2-3yr', '3-4yr', '4-5yr', '5-6yr'])
df['MonthlyToTotalRatio'] = df['MonthlyCharges'] / df['TotalCharges']
df['ServiceCount'] = (df[['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                         'TechSupport', 'StreamingTV', 'StreamingMovies']] == 1).sum(axis=1)

In [79]:
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 
               'PaperlessBilling', 'Churn']
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

In [80]:
cat_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity',
            'OnlineBackup', 'DeviceProtection', 'TechSupport',
            'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

In [81]:
y = 'Churn'
other_columns = [col for col in df.columns if col != y]
new_order = other_columns + [y]
df = df[new_order]

In [82]:
tenure_mapping = {
    '0-1yr': 0,
    '1-2yr': 1, 
    '2-3yr': 2,
    '3-4yr': 3,
    '4-5yr': 4,
    '5-6yr': 5
}
df['TenureGroup'] = df['TenureGroup'].map(tenure_mapping)

In [83]:
print(f"Total NaN: {df.isnull().sum()}")

# Show which columns
nan_cols = df.columns[df.isnull().any()].tolist()
print(f"Columns with NaN: {nan_cols}")

Total NaN: gender                                    0
SeniorCitizen                             0
Partner                                   0
Dependents                                0
tenure                                    0
PhoneService                              0
PaperlessBilling                          0
MonthlyCharges                            0
TotalCharges                             11
TenureGroup                              11
MonthlyToTotalRatio                      11
ServiceCount                              0
MultipleLines_No phone service            0
MultipleLines_Yes                         0
InternetService_Fiber optic               0
InternetService_No                        0
OnlineSecurity_No internet service        0
OnlineSecurity_Yes                        0
OnlineBackup_No internet service          0
OnlineBackup_Yes                          0
DeviceProtection_No internet service      0
DeviceProtection_Yes                      0
TechSupport_No intern

In [84]:
df.dropna(inplace=True)

In [86]:
df.to_csv('../data/processed/telco_dataset.csv', index=False)